In [ ]:
# ==============================================================================
# Sector Rotation Regime Detection Model — Modular Sub-package Implementation
# ==============================================================================

import os
import sys
import warnings
warnings.filterwarnings('ignore')

# ── Colab Setup ───────────────────────────────────────────────────────────
IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    print("Running on Google Colab. Installing dependencies...")
    # !pip install bt pandas_ta kaleido plotly hmmlearn yfinance scikit-learn
    
    # If you haven't cloned the repo, you might need to handle that here.
    # Assuming the repo is in /content/FINRL
    repo_path = "/content/FINRL"
    if repo_path not in sys.path:
        sys.path.append(repo_path)
else:
    # Local path adjustment
    project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
    if project_root not in sys.path:
        sys.path.append(project_root)

import pandas as pd
try:
    from lib.regime_detection.src.constants import *
    from lib.regime_detection.src.data.loader import download_all, _slice_data
    from lib.regime_detection.src.execution.backtest import train_all_sectors, decode_test_sectors, build_weight_matrix, run_bt_backtest
    from lib.regime_detection.src.execution.walk_forward import run_walk_forward
    from lib.regime_detection.src.utils.plotting import plot_results, print_stats
except ImportError as e:
    print(f"Import failed: {e}")
    print("Current sys.path:", sys.path)
    raise e

def main():
    all_tickers = SECTORS + [BENCHMARK]

    if WALK_FORWARD:
        all_data = download_all(all_tickers, WF_FULL_START, WF_FULL_END)
        (wf_weights, wf_decoded, wf_windows, all_close, bench_series) = run_walk_forward(all_data)
        result, close_prices, weights_aligned = run_bt_backtest(
            wf_weights, all_close, bench_series,
            label=f"HMM Walk-Forward ({WF_MODE})"
        )
        print_stats(result)
        mode_label = f"Walk-Forward {WF_MODE.capitalize()} | train={WF_TRAIN_DAYS}d oos={WF_OOS_DAYS}d"
        plot_results(result, weights_aligned, wf_decoded, close_prices, "sector_rotation_wf.png", wf_windows=wf_windows, mode_label=mode_label)

    else:
        all_data = download_all(all_tickers, TRAIN_START, TEST_END)
        train_data = _slice_data(all_data, TRAIN_START, TRAIN_END)
        test_data = _slice_data(all_data, TEST_START, TEST_END)
        benchmark_test = test_data.pop(BENCHMARK, None)
        if benchmark_test is None: raise RuntimeError("Benchmark SPY missing.")
        
        trained = train_all_sectors(train_data)
        decoded_test = decode_test_sectors(test_data, trained)
        weights = build_weight_matrix(decoded_test)
        close_prices = pd.DataFrame({t: df["Close"] for t, df in test_data.items() if t in weights.columns}).ffill().dropna(how="all")
        bench_series = benchmark_test["Close"].reindex(close_prices.index).ffill()
        
        result, close_prices, weights_aligned = run_bt_backtest(weights, close_prices, bench_series, label="HMM Sector Rotation")
        print_stats(result)
        mode_label = f"Train {TRAIN_START}→{TRAIN_END} | Test {TEST_START}→{TEST_END}"
        plot_results(result, weights_aligned, decoded_test, close_prices, "sector_rotation_results.png", mode_label=mode_label)

    print("\nDone!")
    return result

if __name__ == "__main__":
    result = main()